In [1]:
import scanpy as sc
import mygene
import torch
import numpy as np
import pandas as pd
import sys
sys.path.append('/home/cavin/jt/benchmark/raw_code/ominicell/omnicell')  # Adjust this path to where Nicheformer is located
from OmniCell.utils.symbol_ensemble import convert_var_names_to_ensembl, create_gene_mapping_dicts
from OmniCell.loader.CellEmbedding import CellFormer
from task.Cluster.cell_embedding_task import evaluate_cell_embedding

checkpoint_dir = "/home/cavin/jt/benchmark/models/ominicell/OmniCell-v1"
vocab_path = "/home/cavin/jt/benchmark/raw_code/ominicell/omnicell/OmniCell/vocab/Vocabulary.json"


import os
import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps




def gene2ensembl(gene_list,mapping_path):
    mapping_df = pd.read_csv(mapping_path,header=0)
    mapping_df_unique = mapping_df.drop_duplicates(subset=['Gene name'], keep='first')
    mapping_dict = dict(zip(mapping_df_unique['Gene name'], mapping_df_unique['ensembl ID']))
    mapped_gene_list = []
    unmapped_genes = []
    for gene in gene_list:
        if gene.startswith('ENS'):
            mapped_gene_list.append(gene)
            continue
        if gene in mapping_dict:
            mapped_gene_list.append(mapping_dict[gene])
        else:
            mapped_gene_list.append(gene)
            unmapped_genes.append(gene)
    if unmapped_genes:
        print(f"Warning: {len(unmapped_genes)} genes not found in mapping file: {unmapped_genes[:10]}{'...' if len(unmapped_genes) > 10 else ''}")
    return mapped_gene_list
    


def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/batch_integrate"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            save_file_name = "record.csv"
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time / 60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            return result
        return wrapper
    return decorator


In [2]:
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
adata = sc.read_h5ad(simple_path)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [3]:
adata.var_names

Index(['ABCC11', 'ACTA2', 'ACTG2', 'ADAM9', 'ADGRE5', 'ADH1B', 'ADIPOQ',
       'AGR3', 'AHSP', 'AIF1',
       ...
       'TUBB2B', 'TYROBP', 'UCP1', 'USP53', 'VOPP1', 'VWF', 'WARS', 'ZEB1',
       'ZEB2', 'ZNF562'],
      dtype='object', length=313)

In [4]:
adata_idlist = adata.var.ensemble_id.values
adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)
adata.var["gene_name"] = adata.var.index
adata.var.index = adata_idlist
adata.var_names.name = None
adata.var_names

Index(['ENSG00000121270', 'ENSG00000107796', 'ENSG00000163017',
       'ENSG00000168615', 'ENSG00000123146', 'ENSG00000196616',
       'ENSG00000181092', 'ENSG00000173467', 'ENSG00000169877',
       'ENSG00000204472',
       ...
       'ENSG00000137285', 'ENSG00000011600', 'ENSG00000109424',
       'ENSG00000145390', 'ENSG00000154978', 'ENSG00000110799',
       'ENSG00000140105', 'ENSG00000148516', 'ENSG00000169554',
       'ENSG00000171466'],
      dtype='object', length=313)

In [5]:
@measure_resources(task_des="OmiciCell run xenium human breast cancer")
def get_emb():
    cellformer = CellFormer(checkpoint_dir=checkpoint_dir,
            dtype=torch.bfloat16,
            batch_size=16,
            vocab_path=vocab_path,
            n_genes=2000,
            mode="sc",
            threshold=0.9)
    cell_embedding, _  = cellformer.infer(adata)
    return cell_embedding

In [6]:
emb = get_emb()

Loaded parameter: embedder.gene_embedding.weight
Loaded parameter: embedder.value_embedding.shared_experts.0.weight
Loaded parameter: embedder.value_embedding.shared_experts.1.weight
Loaded parameter: embedder.value_embedding.shared_experts.2.weight
Loaded parameter: embedder.value_embedding.shared_experts.3.weight
Loaded parameter: embedder.value_embedding.shared_experts.4.weight
Loaded parameter: embedder.value_embedding.routing_experts.0.weight
Loaded parameter: embedder.value_embedding.routing_experts.1.weight
Loaded parameter: embedder.value_embedding.routing_experts.2.weight
Loaded parameter: embedder.value_embedding.routing_experts.3.weight
Loaded parameter: embedder.value_embedding.routing_experts.4.weight
Loaded parameter: embedder.value_embedding.routing_experts.5.weight
Loaded parameter: embedder.value_embedding.routing_experts.6.weight
Loaded parameter: embedder.value_embedding.routing_experts.7.weight
Loaded parameter: embedder.value_embedding.routing_experts.8.weight
Load

/home/cavin/anaconda3/envs/ominicell/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:151: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": "seurat_v3"}
Processing cells: 100%|██████████| 281780/281780 [00:03<00:00, 78027.80it/s] 


📊 Single Cell dataset: 281780 cells


Inferencing: 100%|██████████| 17612/17612 [05:09<00:00, 56.84batch/s]


✅ Inference complete, total embedding dimension: (281780, 316), expression value dimension: (281780, 313)

RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-04 22:49:12
Target GPU ID               : 0
Execution Time (Minutes)    : 5.4244
Execution Time (Seconds)    : 325.47
CPU Peak Memory Usage (GB)  : 7.0190
GPU Peak Allocated Mem (GB) : 0.2675
GPU Peak Cached Mem (GB)    : 0.3691
CUDA Available              : True

文件已存在，数据已成功追加至末尾。


In [7]:
emb.shape

(281780, 316)

In [8]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"ominicell_dim_{i}" for i in range(emb.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/ominicell_human_breast_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

In [9]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [10]:
adata.obsm["X_OminiCell"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_OminiCell'